# 06 — Building the filing index and joining it to Compustat

**Purpose.** This notebook turns the 394 downloaded containers into one table:
one row per 10-K filing, carrying the CIK, the exact form type, the fiscal
period, the SEC.gov URL the Extractor takes, and the GVKEY the filing belongs
to. Nothing is unzipped to disk — only each filing's `metadata.json` is read out
of the archive, which is a few hundred bytes against a document of a few
megabytes.

This is where the sample is defined, so the choices here are the ones that
matter most in this stage.

**The sample.** `crsp_compustat_link.dta` filtered to the usable-link
convention — `LINKTYPE in (LC, LU)` and `LINKPRIM in (P, C)` — leaves 28,815
spell rows covering **25,207 CIKs and 25,207 GVKEYs, a perfect bijection**. That
bijection is what makes attribution simple: a filing maps to a firm by CIK
alone, with no date logic needed for identity.

Two traps in that filter, both measured:

* `LINKPRIM` must be `P` **or** `C`. Taking `P` alone silently drops 4,446
  firms — 17.6 percent of the sample.
* `LINKENDDT` is `NaT` on 6,040 rows and that is not missing data, it is an open
  link. Coerce it to the vintage end date; dropping those rows deletes the
  survivors.

**The date is for the panel, not for identity.** `LINKDT`/`LINKENDDT` decide
whether a filing's fiscal period falls inside the firm's CRSP-Compustat link
window, which is a question about panel membership. It is not a question about
which firm filed the document, and a filing whose link window has closed is
still that firm's filing. So: attribute by CIK, gate the panel by date, and keep
both columns.

**Fiscal year follows Compustat's convention** — a period ending in January
through May belongs to the previous fiscal year — so that a later merge on
`(GVKEY, fyear)` lines up with the fundamentals.

**Form types.** The bulk dataset carries eight variants. Only five are annual
reports: `10-K`, `10-K405`, `10KSB` (unhyphenated, which is easy to get wrong),
`10KSB40`, and `10-KT`. The other three are amendments. Restricting to plain
`10-K` alone drops **6.9 percent** of in-window annual reports, concentrated in
2000-2008 and among small filers.

**Inputs.**

| Source | Path |
|---|---|
| Containers | `SEC_BULK / {year}/{year}-{month}.zip` — from `07` |
| Link table | `data/raw/01_firm_id/crsp_compustat_link.dta` |
| S&P 500 membership | `data/raw/01_firm_id/sp500_constituents.dta` |

**Outputs.**

| File | Path |
|---|---|
| Filing index | `SEC_RAW / filings.parquet` — one row per filing, with GVKEY, fiscal year, S&P 500 flag |
| Sample frame | `SEC_RAW / sample_firms.parquet` — one row per CIK, with GVKEY and S&P 500 flag |

## Setup

In [ ]:
import json
import sys
import zipfile

sys.path.insert(0, "../lib")

import pandas as pd

from paths import DATA_REPO, SEC_BULK, SEC_RAW

FIRM_ID = DATA_REPO / "raw" / "01_firm_id"

# The five annual-report form types. `10KSB` and `10KSB40` carry no hyphen in
# EDGAR; writing them as `10-KSB` matches nothing and loses every small-business
# filer without raising anything.
ANNUAL_FORMS = ("10-K", "10-K405", "10KSB", "10KSB40", "10-KT")

# The link file's last vintage. Used to close open-ended LINKENDDT rather than
# dropping them.
VINTAGE_END = pd.Timestamp("2025-12-31")

print(f"containers: {SEC_BULK}")
print(f"annual forms kept: {ANNUAL_FORMS}")

## Step 1 — The sample frame

Three filters, then the S&P 500 flag. The counts printed after each are what a
reader needs to see the cost of every choice, and they are the numbers the
paper's sample description is built from.

In [ ]:
link = pd.read_stata(FIRM_ID / "crsp_compustat_link.dta", convert_categoricals=False)
before = len(link)

link["cik"] = link["cik"].astype(str).str.strip()
link = link.loc[link["cik"].ne("") & link["cik"].ne("nan")]
after_cik = len(link)

link = link.loc[link["LINKTYPE"].isin(["LC", "LU"]) & link["LINKPRIM"].isin(["P", "C"])]
after_link = len(link)

link["LINKDT"] = pd.to_datetime(link["LINKDT"], errors="coerce")
link["LINKENDDT"] = pd.to_datetime(link["LINKENDDT"], errors="coerce").fillna(VINTAGE_END)

print(f"link rows                       {before:>8,}")
print(f"  with a CIK                    {after_cik:>8,}  (dropped {before - after_cik:,})")
print(f"  usable link type and primacy  {after_link:>8,}  (dropped {after_cik - after_link:,})")
print(f"  distinct CIK                  {link['cik'].nunique():>8,}")
print(f"  distinct GVKEY                {link['GVKEY'].nunique():>8,}")

# The bijection is worth asserting rather than assuming: every downstream join
# is one-to-one because of it.
assert link["cik"].nunique() == link["GVKEY"].nunique(), "cik and GVKEY are not in bijection"

# CIK arrives in two incompatible shapes and the difference is silent, so it is
# normalised once, here, and asserted rather than assumed:
#   crsp_compustat_link.dta  "0000320193"  str, zero-padded to 10
#   metadata.json            "320193"      str or int, unpadded
# A padded lookup against sec-api returns total=0 with HTTP 200 — no error,
# indistinguishable from "this firm never filed" — so the join key is the
# unpadded string on both sides.
assert link["cik"].dtype == object, f"cik is {link['cik'].dtype}, expected object"
assert link["cik"].str.fullmatch(r"\d{10}").all(), "not every cik is 10 digits"
link["cik_unpadded"] = link["cik"].str.lstrip("0")
assert link["cik_unpadded"].str.fullmatch(r"[1-9]\d*").all(), "unpadded cik is malformed"

# GVKEY is likewise a zero-padded string, not a number. Casting it to int
# anywhere downstream would silently drop the padding and break a later merge
# against Compustat fundamentals.
assert link["GVKEY"].dtype == object, f"GVKEY is {link['GVKEY'].dtype}, expected object"
print(f"\ncik      dtype={link['cik'].dtype}  e.g. {link['cik'].iloc[0]!r} "
      f"-> unpadded {link['cik_unpadded'].iloc[0]!r}")
print(f"GVKEY    dtype={link['GVKEY'].dtype}  e.g. {link['GVKEY'].iloc[0]!r} "
      f"(width {link['GVKEY'].str.len().unique().tolist()})")

## Step 2 — S&P 500 membership

The constituents file is a daily panel keyed on `PERMNO`, not on CIK or GVKEY,
so membership reaches the sample through the link table's `LPERMNO`. Firms that
were ever in the index are flagged; this is a membership-ever flag, not a
point-in-time one, and it is used only to decide extraction order.

In [ ]:
sp = pd.read_stata(FIRM_ID / "sp500_constituents.dta", convert_categoricals=False,
                   columns=["PERMNO"])
sp500_permnos = set(sp["PERMNO"].dropna().astype(int))

link["permno"] = pd.to_numeric(link["LPERMNO"], errors="coerce")
link["in_sp500"] = link["permno"].isin(sp500_permnos)

firms = (
    link.groupby("cik", as_index=False)
    .agg(gvkey=("GVKEY", "first"),
         cik_unpadded=("cik_unpadded", "first"),
         conm=("conm", "first"),
         tic=("tic", "first"),
         sic=("sic", "first"),
         permno=("permno", "first"),
         in_sp500=("in_sp500", "any"),
         link_start=("LINKDT", "min"),
         link_end=("LINKENDDT", "max"))
)
print(f"sample firms      {len(firms):>7,}")
print(f"  ever in S&P 500 {int(firms['in_sp500'].sum()):>7,}")
print(f"  matched PERMNOs {firms['permno'].isin(sp500_permnos).sum():>7,} "
      f"of {len(sp500_permnos):,} index PERMNOs")

## Step 3 — Read every filing's metadata out of the containers

`metadata.json` only. Reading the primary document here would mean unzipping 34
GB to build a table of a few hundred thousand rows, and the document is not
needed until extraction — which fetches from SEC.gov by URL anyway.

In [ ]:
records = []
containers = sorted(SEC_BULK.rglob("*.zip"))
print(f"{len(containers)} containers on disk")

for n, path in enumerate(containers, 1):
    with zipfile.ZipFile(path) as z:
        for name in z.namelist():
            if not name.endswith("metadata.json"):
                continue
            meta = json.loads(z.read(name))
            # metadata.json carries cik as a string in some containers and as a
            # JSON number in others; str() before lstrip covers both.
            records.append({
                "accessionNo": meta.get("accessionNo"),
                "cik_unpadded": str(meta.get("cik", "")).strip().lstrip("0"),
                "formType": meta.get("formType"),
                "filedAt": meta.get("filedAt"),
                "periodOfReport": meta.get("periodOfReport"),
                "companyName": meta.get("companyName"),
                "ticker": meta.get("ticker") or None,
                "filingUrl": meta.get("linkToFilingDetails"),
                "container": path.relative_to(SEC_BULK).as_posix(),
            })
    if n % 100 == 0:
        print(f"  {n}/{len(containers)} containers, {len(records):,} filings")

filings = pd.DataFrame.from_records(records)
print(f"\n{len(filings):,} filings read")

# The join key has to be the same shape on both sides. Check it before merging,
# because a shape mismatch here produces an empty merge, not an error.
bad = filings.loc[~filings["cik_unpadded"].str.fullmatch(r"[1-9]\d*", na=False)]
print(f"filings whose cik is not a clean unpadded number: {len(bad):,}")
if len(bad):
    print(bad[["accessionNo", "cik_unpadded"]].head().to_string(index=False))
print(f"cik_unpadded dtype={filings['cik_unpadded'].dtype}  "
      f"e.g. {filings['cik_unpadded'].iloc[0]!r}")
print()
print(filings["formType"].value_counts().to_string())

## Step 4 — Restrict to annual reports and join to the sample

Two restrictions, each printed. The form-type filter is an exact string match
against `ANNUAL_FORMS` — the bulk metadata carries the true form type, so
nothing here depends on a search analyser the way a Query API filter would.

In [ ]:
before_rows = len(filings)
filings = filings.loc[filings["formType"].isin(ANNUAL_FORMS)]
after_forms = len(filings)

# Both sides are unpadded strings by construction above. An inner merge that
# returns almost nothing is the signature of a key-shape mismatch, so the match
# rate is printed rather than inferred from the row count alone.
key_overlap = len(set(filings["cik_unpadded"]) & set(firms["cik_unpadded"]))
print(f"  distinct CIKs: {filings['cik_unpadded'].nunique():,} in the archive, "
      f"{len(firms):,} in the sample, {key_overlap:,} shared")

filings = filings.merge(
    firms[["cik_unpadded", "cik", "gvkey", "conm", "in_sp500"]],
    on="cik_unpadded", how="inner", validate="many_to_one",
)
after_sample = len(filings)

print(f"filings in the archive      {before_rows:>8,}")
print(f"  annual report form types  {after_forms:>8,}  (dropped {before_rows - after_forms:,})")
print(f"  in the Compustat sample   {after_sample:>8,}  (dropped {after_forms - after_sample:,})")
print(f"  distinct firms            {filings['gvkey'].nunique():>8,}")
print(f"  S&P 500 filings           {int(filings['in_sp500'].sum()):>8,}")

## Step 5 — Fiscal year and panel membership

`periodOfReport` arrives as JSON `null` when absent, not as an empty string, so
it is coerced rather than stripped. A filing without one falls back to
`filedAt` minus 90 days and is flagged; those rows should be excluded from an
analysis panel by default, which is why the flag travels with the row rather
than being applied here.

In [ ]:
filings["filedAt"] = pd.to_datetime(filings["filedAt"], format="mixed", utc=True)
period = pd.to_datetime(filings["periodOfReport"], errors="coerce", utc=True)

filings["period_imputed"] = period.isna()
period = period.fillna(filings["filedAt"] - pd.Timedelta(days=90))
filings["periodEnd"] = period.dt.tz_localize(None)

# Compustat convention: a period ending Jan-May belongs to the prior fiscal year.
filings["fyear"] = filings["periodEnd"].dt.year.where(
    filings["periodEnd"].dt.month > 5, filings["periodEnd"].dt.year - 1
)

# Panel membership against the spell rows, not a min/max collapse. Collapsing
# fills the histories that have gaps and overstates coverage.
spells = link[["cik", "LINKDT", "LINKENDDT"]]
paired = filings[["accessionNo", "cik", "periodEnd"]].merge(spells, on="cik", how="left")
in_window = (
    (paired["periodEnd"] >= paired["LINKDT"]) & (paired["periodEnd"] <= paired["LINKENDDT"])
).groupby(paired["accessionNo"]).any()
filings["in_link_window"] = filings["accessionNo"].map(in_window).fillna(False)

print(f"period imputed from filedAt : {int(filings['period_imputed'].sum()):,} "
      f"({filings['period_imputed'].mean():.2%})")
print(f"inside the CRSP-Compustat link window : {int(filings['in_link_window'].sum()):,} "
      f"({filings['in_link_window'].mean():.1%})")
print()
print(filings.groupby(filings["fyear"] // 5 * 5).size().rename("filings").to_string())

## Step 5b — Repairing the filing URL

The Extractor takes a URL and resolves the document behind it, so this is the one
field in the index that has to be exactly right. For **25,675 filings — 16.2
percent of the sample — sec-api's own metadata ships a URL that cannot work**:
`linkToFilingDetails` is a bare CIK directory,

    https://www.sec.gov/Archives/edgar/data/1000180/

which identifies a *company*, not a filing. The Extractor cannot resolve a
document from it and answers `HTTP 404 {"error":"filing type not supported"}`.

That message is misleading, and reading it literally cost a full extraction run.
The filing type is fine; the link is degenerate. Every affected filing was filed
on or before **1999-05-21**, which is why the failures first looked like an
old-filings problem — but vintage is a correlate, not the cause. Filings from the
same era that happen to carry a proper URL extract perfectly, so keying the rule
on `fyear` would misclassify them. The predicate is about URL shape only.

The repair rebuilds the URL from the accession number, which this index already
holds. Verified directly on twelve randomly drawn affected filings: **12 of 12
returned 404 on the shipped URL, and all 12 returned real Item 1 text on the
rebuilt one**, ranging from 11,791 to 201,358 characters. A six-filing control on
untouched URLs still returned text, so the repair does not disturb the rest.

In [ ]:
import re

# A URL with nothing after the CIK. Any second path segment — an accession
# directory or a document filename — resolves, so the predicate is about URL
# shape, not about the filing's age or form type.
BARE_CIK = re.compile(r"^https://www\.sec\.gov/Archives/edgar/data/\d+/?$")

filings["url_bare"] = filings["filingUrl"].fillna("").str.match(BARE_CIK)


def accession_url(cik: str, accession: str) -> str:
    """The EDGAR accession directory, which the Extractor can resolve."""
    return f"https://www.sec.gov/Archives/edgar/data/{cik}/{accession.replace('-', '')}/"


filings["filingUrl_original"] = filings["filingUrl"]
filings.loc[filings["url_bare"], "filingUrl"] = [
    accession_url(c, a)
    for c, a in zip(filings.loc[filings["url_bare"], "cik_unpadded"],
                    filings.loc[filings["url_bare"], "accessionNo"])
]

n_bare = int(filings["url_bare"].sum())
print(f"filing URLs that were a bare CIK directory : {n_bare:,} ({n_bare / len(filings):.1%})")
if n_bare:
    filed = pd.to_datetime(filings.loc[filings["url_bare"], "filedAt"], utc=True, format="mixed")
    print(f"  latest filedAt among them               : {filed.max().date()}")
    print(f"  of these, fiscal year >= 2000           : "
          f"{int(filings.loc[filings['url_bare'], 'fyear'].ge(2000).sum()):,}")
    ex = filings.loc[filings["url_bare"]].iloc[0]
    print("  example repair:")
    print(f"    was {ex['filingUrl_original']}")
    print(f"    now {ex['filingUrl']}")

# A survivor in the degenerate shape is a guaranteed 404, so this is asserted.
left = int(filings["filingUrl"].fillna("").str.match(BARE_CIK).sum())
assert left == 0, f"{left} filing URLs are still a bare CIK directory"
print("\nno bare-CIK URLs remain in the index")

## Step 6 — Persist

In [ ]:
SEC_RAW.mkdir(parents=True, exist_ok=True)
filings = filings.sort_values(["cik_unpadded", "periodEnd"]).reset_index(drop=True)
filings.to_parquet(SEC_RAW / "filings.parquet", index=False)
firms.to_parquet(SEC_RAW / "sample_firms.parquet", index=False)

print(f"filings.parquet      {len(filings):>8,} rows")
print(f"sample_firms.parquet {len(firms):>8,} rows")
print()
sp = filings.loc[filings["in_sp500"]]
print(f"S&P 500 subset: {len(sp):,} filings across {sp['gvkey'].nunique():,} firms "
      f"-> {len(sp) * 6:,} extractor calls")
print(f"full sample   : {len(filings):,} filings -> {len(filings) * 6:,} extractor calls")

## Verification

Five checks, each naming the failure it catches.

1. **`accessionNo` is unique.** A duplicate means the same filing was read from
   two containers, and every count taken over this table would double it.
2. **Every row has a filing URL.** That URL is the only input the Extractor
   takes; a null would fail inside the extraction loop rather than here.
3. **`cik_unpadded` never has a leading zero.** The padded form returns
   `total=0` from sec-api with HTTP 200 — silent, and indistinguishable from a
   firm that never filed.
4. **Every filing joins to exactly one GVKEY.** The bijection asserted in step 1
   should make this automatic; checking it here catches a merge that fanned out.
5. **Form types are exactly the five annual ones.** Catches an amendment
   surviving the filter, which would put two documents in one firm-year.

In [ ]:
dupes = int(filings["accessionNo"].duplicated().sum())
assert dupes == 0, f"{dupes} duplicate accessionNo — a filing was read twice"
print(f"1) {len(filings):,} filings, all accessionNo distinct")

no_url = int(filings["filingUrl"].isna().sum())
assert no_url == 0, f"{no_url} filings have no linkToFilingDetails"
bare_left = int(filings["filingUrl"].str.match(BARE_CIK).sum())
assert bare_left == 0, f"{bare_left} filings still carry a bare-CIK URL"
print(f"2) every filing carries a resolvable extractor URL "
      f"({int(filings['url_bare'].sum()):,} of them repaired)")

padded = int(filings["cik_unpadded"].str.startswith("0").sum())
assert padded == 0, f"{padded} CIKs still carry a leading zero"
print("3) no padded CIKs — sec-api would return a silent zero for these")

per_filing = filings.groupby("accessionNo")["gvkey"].nunique()
assert (per_filing == 1).all(), f"{int((per_filing > 1).sum())} filings map to >1 GVKEY"
print("4) every filing maps to exactly one GVKEY")

extra = set(filings["formType"]) - set(ANNUAL_FORMS)
assert not extra, f"unexpected form types survived the filter: {extra}"
print(f"5) form types are exactly {sorted(set(filings['formType']))}")

## Step 8 — Does the mapping actually hold? A name check, and ten rows to read

Every check above is internal: they confirm the join did what the code said, not
that the result is *right*. A key-shape bug would pass all five and still attach
each filing to the wrong firm.

The independent check is the company name. SEC's `companyName` comes from the
filing header; Compustat's `conm` comes from a different vendor entirely. They
are never byte-identical — "APPLE INC" against "Apple Inc." — but if the join is
correct they agree once case, punctuation and the usual suffixes are stripped.
If the join is wrong they are unrelated strings, and no amount of normalising
makes them match.

Then ten filings drawn at random, printed side by side, because an aggregate
agreement rate can hide a systematic offset that a human reading ten rows
catches immediately.

In [ ]:
import re

SEED = 20260820
SUFFIXES = re.compile(
    r"(inc|incorporated|corp|corporation|co|company|ltd|limited|llc|lp|plc|"
    r"holdings?|group|the|new|cl [ab]|-cl [ab]|adr|sa|nv|ag)")


def normalise(name: str) -> str:
    """Case, punctuation and corporate suffixes removed — what is left is the name."""
    s = str(name).lower()
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    s = SUFFIXES.sub(" ", s)
    return re.sub(r"\s+", " ", s).strip()


# keep="last" after the sort by (cik, periodEnd) means each firm is represented
# by its MOST RECENT filing. That is the right comparison: `conm` carries the
# name Compustat holds today, so pairing it with a 1994 filing header measures
# thirty years of renaming rather than the quality of the join. Both rates are
# reported below to make the size of that effect visible.
check = filings.drop_duplicates("cik_unpadded", keep="last").copy()
oldest = filings.drop_duplicates("cik_unpadded", keep="first").copy()
check["sec_name"] = check["companyName"].map(normalise)
check["cs_name"] = check["conm"].map(normalise)

# Agreement is generous on purpose: one name being a prefix of the other covers
# "AMERICAN AIRLINES GROUP" against "American Airlines Group Inc", and the token
# overlap covers word-order and abbreviation differences.
def agrees(row) -> bool:
    a, b = row["sec_name"], row["cs_name"]
    if not a or not b:
        return False
    if a == b or a.startswith(b) or b.startswith(a):
        return True
    ta, tb = set(a.split()), set(b.split())
    return bool(ta & tb) and len(ta & tb) / min(len(ta), len(tb)) >= 0.5


check["name_agrees"] = check.apply(agrees, axis=1)
rate = check["name_agrees"].mean()

oldest["sec_name"] = oldest["companyName"].map(normalise)
oldest["cs_name"] = oldest["conm"].map(normalise)
rate_oldest = oldest.apply(agrees, axis=1).mean()

print(f"company names agree on {check['name_agrees'].sum():,} of {len(check):,} "
      f"mapped firms ({rate:.1%}), comparing each firm's LATEST filing header")
print(f"  same test on each firm's EARLIEST filing header: {rate_oldest:.1%}")
print("  The gap between the two is renaming over the sample period, not join error.")
print()
print("--- ten random mapped filings, read these ---")
sample = filings.sample(10, random_state=SEED)
for row in sample.itertuples():
    ok = "OK " if agrees({"sec_name": normalise(row.companyName),
                          "cs_name": normalise(row.conm)}) else "?? "
    print(f"  [{ok}] cik {row.cik_unpadded:>8} (padded {row.cik})  gvkey {row.gvkey}"
          f"  fyear {row.fyear}  {row.formType}")
    print(f"        SEC filing header : {row.companyName}")
    print(f"        Compustat conm    : {row.conm}")
print()
disagree = check.loc[~check["name_agrees"]]
if len(disagree):
    print(f"--- {len(disagree):,} firms whose names do not agree (first 8) ---")
    for row in disagree.head(8).itertuples():
        print(f"  cik {row.cik_unpadded:>8}  SEC {row.companyName!r} vs CS {row.conm!r}")
    print("  Name disagreement is not automatically an error — firms are renamed,")
    print("  acquired and re-registered, and Compustat carries the latest name while")
    print("  the filing header carries the name at filing time. What would be an")
    print("  error is a LOW overall rate; a handful of these is expected.")

# A wrong join key produces near-zero agreement, so this is the real gate.
assert rate > 0.80, (
    f"only {rate:.1%} of mapped firms have agreeing names — the CIK join is "
    "probably wrong, most likely a padding or dtype mismatch"
)
print(f"\n6) name agreement {rate:.1%} — the CIK-to-GVKEY join holds")